# CIFAR-10 Image Classification with PyTorch + timm

Notebook goal: compare a simple CNN baseline against a modern timm backbone on CIFAR-10 using CUDA by default, with augmentation, confusion matrix, and error analysis.


## 2. Project overview

We train and compare two models:

- Baseline: custom SimpleCNN
- Main model: timm ConvNeXt-Tiny (pretrained)

Both are trained on the same CIFAR-10 split and evaluated with identical metrics.


## 3. Learning objectives

- Load CIFAR-10 with robust transforms
- Use Albumentations for augmentation
- Train PyTorch models with GPU auto-detection
- Fine-tune a timm backbone
- Compare baseline vs main model with quantitative and qualitative analysis


## 4. Problem statement

Classify each 32x32 RGB image into one of 10 CIFAR-10 classes.


## 5. Why this project matters

This project demonstrates the practical lift from a classic CNN baseline to a modern pretrained vision backbone while keeping the pipeline educational and reproducible.


## 6. Dataset overview

- Dataset: torchvision.datasets.CIFAR10
- Classes: 10
- Images: 50,000 train, 10,000 test
- Resolution: 32x32 RGB


## 7. Dataset source and license notes

Source: CIFAR-10 from the University of Toronto.
Use: educational and research contexts.
Known limits: low resolution and relatively simple class taxonomy compared to real production datasets.


## 8. Environment setup


In [ ]:
import importlib
import subprocess
import sys

packages = [
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("timm", "timm"),
    ("albumentations", "albumentations"),
    ("scikit-learn", "sklearn"),
    ("seaborn", "seaborn")
]

for pkg, module_name in packages:
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Dependencies ready.")


## 9. Imports


In [ ]:
import os
import json
import random
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision.datasets import CIFAR10

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report


## 10. Configuration / constants


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 128
EPOCHS_BASELINE = 8
EPOCHS_TIMM = 8
LR_BASELINE = 1e-3
LR_TIMM = 3e-4
NUM_WORKERS = 0
NUM_CLASSES = 10

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB): {props.total_memory / 1e9:.2f}")
print(f"Seed: {SEED}")


## 11. Dataset download and loading


In [ ]:
train_raw = CIFAR10(root="./data", train=True, download=True)
test_raw = CIFAR10(root="./data", train=False, download=True)

print(f"Train raw size: {len(train_raw)}")
print(f"Test raw size: {len(test_raw)}")


## 12. Data validation checks


In [ ]:
assert len(train_raw) == 50000, "Unexpected CIFAR-10 train size"
assert len(test_raw) == 10000, "Unexpected CIFAR-10 test size"

sample_img, sample_label = train_raw[0]
print(f"Sample image size: {sample_img.size}")
print(f"Sample label index: {sample_label}, class: {CLASS_NAMES[sample_label]}")

labels = np.array(train_raw.targets)
class_counts = {CLASS_NAMES[i]: int((labels == i).sum()) for i in range(NUM_CLASSES)}
print("Class counts (train):", class_counts)


## 13. Exploratory data analysis


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for class_idx in range(NUM_CLASSES):
    idx = int(np.where(np.array(train_raw.targets) == class_idx)[0][0])
    img, _ = train_raw[idx]
    ax = axes[class_idx // 5, class_idx % 5]
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[class_idx])
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "cifar10_class_samples.png"), dpi=120)
plt.show()


## 14. Train/validation/test split strategy

We split official training data into train/validation (90/10) with a fixed seed.


## 15. Preprocessing and augmentation strategy

- Albumentations train pipeline: flip, shift/scale/rotate, coarse dropout, normalize.
- Validation/test: normalize only.


In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_tfms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.CoarseDropout(max_holes=1, max_height=8, max_width=8, p=0.3),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

val_test_tfms = A.Compose([
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

class CIFAR10Albumentations(Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        img_np = np.array(img)
        transformed = self.transform(image=img_np)
        return transformed["image"], label


## 16. Baseline approach

SimpleCNN with three convolution blocks and a small classifier head.


In [ ]:
train_base = CIFAR10(root="./data", train=True, download=False)
test_base = CIFAR10(root="./data", train=False, download=False)

train_size = int(0.9 * len(train_base))
val_size = len(train_base) - train_size
train_subset, val_subset = random_split(train_base, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

train_ds = CIFAR10Albumentations(train_subset, train_tfms)
val_ds = CIFAR10Albumentations(val_subset, val_test_tfms)
test_ds = CIFAR10Albumentations(test_base, val_test_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_ds), len(val_ds), len(test_ds))


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


## 17. Main model/workflow

Modern backbone: timm ConvNeXt-Tiny with pretrained weights and CIFAR-10 output head.


In [ ]:
def build_timm_model(num_classes=10):
    model = timm.create_model("convnext_tiny", pretrained=True, num_classes=num_classes)
    return model


## 18. Training loop or fine-tuning pipeline


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    preds_all = []
    labels_all = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds_all.extend(torch.argmax(logits, dim=1).detach().cpu().numpy().tolist())
        labels_all.extend(y.detach().cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(labels_all, preds_all)
    return avg_loss, acc

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds_all = []
    labels_all = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        preds_all.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
        labels_all.extend(y.cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(labels_all, preds_all)
    return avg_loss, acc, np.array(labels_all), np.array(preds_all)

def fit_model(model, name, epochs, lr):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_path = os.path.join(SAVE_DIR, f"best_{name}.pth")

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), best_path)

        print(f"{name} | Epoch {epoch}/{epochs} | train_acc={tr_acc:.4f} | val_acc={va_acc:.4f}")

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, history


In [ ]:
baseline_model = SimpleCNN(num_classes=NUM_CLASSES)
timm_model = build_timm_model(num_classes=NUM_CLASSES)

baseline_model, hist_baseline = fit_model(baseline_model, "simplecnn", EPOCHS_BASELINE, LR_BASELINE)
timm_model, hist_timm = fit_model(timm_model, "timm_convnext_tiny", EPOCHS_TIMM, LR_TIMM)


## 19. Inference examples


In [ ]:
@torch.no_grad()
def predict_batch(model, loader, max_items=8):
    model.eval()
    x, y = next(iter(loader))
    x = x.to(DEVICE)
    logits = model(x)
    preds = torch.argmax(logits, dim=1).cpu().numpy()
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    return x.cpu(), y.numpy(), preds, probs

x_show, y_true_show, y_pred_show, y_prob_show = predict_batch(timm_model, test_loader)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    if i >= len(y_true_show):
        ax.axis("off")
        continue
    img = x_show[i].permute(1, 2, 0).numpy()
    img = (img * np.array(std) + np.array(mean)).clip(0, 1)
    ax.imshow(img)
    ax.set_title(f"T:{CLASS_NAMES[y_true_show[i]]}\nP:{CLASS_NAMES[y_pred_show[i]]}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "inference_examples.png"), dpi=120)
plt.show()


## 20. Evaluation


In [ ]:
criterion_eval = nn.CrossEntropyLoss()

_, _, y_true_base, y_pred_base = evaluate(baseline_model, test_loader, criterion_eval)
_, _, y_true_timm, y_pred_timm = evaluate(timm_model, test_loader, criterion_eval)

acc_base = accuracy_score(y_true_base, y_pred_base)
acc_timm = accuracy_score(y_true_timm, y_pred_timm)

p_base, r_base, f1_base, _ = precision_recall_fscore_support(y_true_base, y_pred_base, average="macro", zero_division=0)
p_timm, r_timm, f1_timm, _ = precision_recall_fscore_support(y_true_timm, y_pred_timm, average="macro", zero_division=0)

print("Baseline  - Acc: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}".format(acc_base, p_base, r_base, f1_base))
print("timm Model- Acc: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}".format(acc_timm, p_timm, r_timm, f1_timm))

metrics = {
    "baseline": {"accuracy": float(acc_base), "precision_macro": float(p_base), "recall_macro": float(r_base), "f1_macro": float(f1_base)},
    "timm": {"accuracy": float(acc_timm), "precision_macro": float(p_timm), "recall_macro": float(r_timm), "f1_macro": float(f1_timm)}
}
with open(os.path.join(SAVE_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics.json")


In [ ]:
cm_base = confusion_matrix(y_true_base, y_pred_base)
cm_timm = confusion_matrix(y_true_timm, y_pred_timm)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm_base, annot=False, cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Baseline SimpleCNN Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(cm_timm, annot=False, cmap="Greens", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("timm ConvNeXt-Tiny Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrices.png"), dpi=120)
plt.show()


## 21. Error analysis


In [ ]:
mis_idx = np.where(y_true_timm != y_pred_timm)[0]
print(f"Total timm misclassifications: {len(mis_idx)}")

print("\nTop error pairs (timm):")
pair_counts = {}
for i in mis_idx:
    key = (CLASS_NAMES[y_true_timm[i]], CLASS_NAMES[y_pred_timm[i]])
    pair_counts[key] = pair_counts.get(key, 0) + 1

sorted_pairs = sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True)[:10]
for (t, p), c in sorted_pairs:
    print(f"{t} -> {p}: {c}")


In [ ]:
raw_test = CIFAR10(root="./data", train=False, download=False)
show_n = min(12, len(mis_idx))
sel = mis_idx[:show_n]

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, ax in enumerate(axes.flatten()):
    if i >= show_n:
        ax.axis("off")
        continue
    idx = int(sel[i])
    img, _ = raw_test[idx]
    ax.imshow(img)
    ax.set_title(f"T:{CLASS_NAMES[y_true_timm[idx]]}\nP:{CLASS_NAMES[y_pred_timm[idx]]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_samples_timm.png"), dpi=120)
plt.show()


## 22. Limitations

- Compact training budget (few epochs) is educational, not benchmark-maximized.
- CIFAR-10 resolution is low and may not transfer directly to high-resolution domains.
- Results can vary by GPU, seed, and package versions.


## 23. How to improve this project

- Train longer with cosine scheduler and warmup.
- Add stronger augmentation (MixUp/CutMix).
- Try larger timm backbones if GPU memory allows.
- Add calibration and confidence-based error review.


## 24. Production considerations

- Export best model with TorchScript/ONNX.
- Add input validation and monitoring for data drift.
- Track class-wise errors continuously in production.


## 25. Common mistakes

- Forgetting to normalize test data exactly like training.
- Comparing models with different splits/transforms.
- Reporting only accuracy without class-wise checks.


## 26. Mini challenge / exercises

1. Replace ConvNeXt-Tiny with a different timm model and compare.
2. Add test-time augmentation and report gain/loss.
3. Analyze the hardest class pair and propose targeted augmentation.


## 27. Final summary / key takeaways

- Baseline CNN provides a grounded reference.
- Modern timm backbone typically improves metrics and error patterns.
- Structured evaluation (confusion matrix + error analysis) is essential beyond headline accuracy.
